# NaN check, missing files check and average structural connectivity

Aggregates structural connectivity data into one averaged structural connectivity matrix from diffusion MRI for all subjects of the depressed and control cohorts separately defined in `00_inital_cohort_selection.ipynb` and `01_cohort_matching.ipynb`. Additionally excludes subjects with missing functional or structural data from the combined cohort file and saves it for future use. The notebook includes steps for loading ROI labels (from `04_merge_aggregate_rfMRI.ipynb`) for visualization, computing the average structural connectome, saving QC CSVs (missing files, NaN values), plotting and saving the average structural connectivity matrix figure, and updating the combined cohort file to exclude subjects with missing data.

## 1) Configuration 
Edit these variables, then run the notebook top-to-bottom.

In [ ]:
from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
project_root = Path(".../subtyping_depression")

cohort_type = "F32"  # 'F32' or 'control'

data_dir = project_root / "data" / "UKB" / f"{cohort_type}_notask_STRCO_RSSCHA_RSTIA"
combined_cohort_file = project_root / "data" / "UKB" / "cohorts" / "combined_cohort_F32.csv"
vis_dir = project_root / "reports" / "figures" / "schaefer1000+tian54" / "structural_con"

labels_path = data_dir / "Schaefer7n1000p_TianSubcortexS4_labels.txt"

connectome_subdir = "i2"
connectome_template = "connectome_streamline_count_10M.csv.gz"

show_progress = True
log_transform = True

# Make local modules importable from the notebook
features_dir = (project_root / "source_code" / "connectivity_matrices").resolve()
if str(features_dir) not in sys.path:
    sys.path.insert(0, str(features_dir))

from avg_strct_utils import (
    compute_average_structural_connectivity,
    list_subject_ids,
    load_labels_txt,
    load_excluded_subject_ids,
    normalize_for_plot,
    plot_connectivity_matrix,
    exclude_subjects_by_eid
)

print("data_dir:", data_dir)
print("combined_cohort_file:", combined_cohort_file)
print("vis_dir:", vis_dir)
print("labels_path exists:", labels_path.exists())
print("features_dir:", features_dir)

## 2) Load ROI labels and subject IDs
This section loads ROI labels and subject IDs for further processing.

In [ ]:
labels = load_labels_txt(labels_path)
excluded_subjects_csv = data_dir / "missing_subjects_resting_state_timeseries.csv"
excluded_subjects_column = "subject_id"

subject_ids = list_subject_ids(data_dir)
print(f"Found {len(subject_ids)} subjects.")
print(subject_ids[:5])

if excluded_subjects_csv.exists():
    excluded = set(load_excluded_subject_ids(excluded_subjects_csv, column=excluded_subjects_column))
    subject_ids = [sid for sid in subject_ids if sid not in excluded]
    subject_ids.sort()
    print(f"After exclusion, {len(subject_ids)} subjects remain.")
    print(subject_ids[:5])

if not subject_ids:
    raise ValueError(f"No subject directories to process under {data_dir}")

n_subjects = len(subject_ids)

## 3) Compute the average structural connectome
This section computes the average structural connectome, skipping subjects whose matrices contain NaNs by default.

In [ ]:
result = compute_average_structural_connectivity(
    data_dir=data_dir,
    subject_ids=subject_ids,
    subdir=connectome_subdir,
    filename_template=connectome_template,
    skip_nan_subjects=True,
    show_progress=show_progress,
)

avg_mat = result.avg_matrix
print("Average connectivity matrix shape:", avg_mat.shape)
print("Candidates:", result.n_candidates)
print("Included:", result.n_included)
print("Missing connectomes:", len(result.missing_subjects))
print("NaN matrices:", len(result.nan_subjects))

## Save outputs (CSV reports + average `.npy`)

In [ ]:
import numpy as np
import pandas as pd

out_nans_csv = data_dir / f"structural_connectivity_matrices_streamline_count_nans.csv"
out_missing_csv = data_dir / "missing_subjects_structural_connectomes.csv"
out_avg_npy = data_dir / f"average_structural_connectivity_matrix_streamline_count_10M.npy"

result.nan_subjects.to_csv(out_nans_csv, index=False)
pd.DataFrame({"subject_id": result.missing_subjects}).to_csv(out_missing_csv, index=False)
np.save(out_avg_npy, result.avg_matrix)

print("Saved NaN report to", out_nans_csv)
print("Saved missing subjects report to", out_missing_csv)
print("Saved average structural connectivity matrix to", out_avg_npy)

## 4) Visualize and save the matrix figure


In [ ]:
vis_dir.mkdir(parents=True, exist_ok=True)
out_fig = vis_dir / f"{cohort_type}_average_structural_connectivity_matrix_streamline_count_10M.svg"

plot_mat = normalize_for_plot(avg_mat, log_transform=log_transform)

plot_connectivity_matrix(
    plot_mat,
    labels=labels,
    title="Average Structural Connectivity Matrix",
    cmap="Reds",
    figure=(20, 20),
    reorder=False,
    out_path=out_fig,
    dpi=300,
)

print("Saved visualization to", out_fig)

## 5) Exclude control and depression subjects

Exclude any subjects from the depression (`00_inital_cohort_selection.ipynb`) and combined cohort (`01_cohort_matching.ipynb`) that do not have functional (generated missing subjects file in `03_merge_aggregate_rfMRI.ipynb`) or structural (generated missing subjects file in this notebook) data. Save this new combined cohort for future notebooks. Make sure that you ran this notebook twice: once for the depression cohort and once for the control cohort, so that both missing subjects files are generated and can be used for exclusion here.

In [ ]:
import pandas as pd
missing_subjects_functional = pd.read_csv(data_dir / f"missing_subjects_resting_state_timeseries.csv")
missing_subjects_structural = pd.read_csv(data_dir / f"missing_subjects_structural_connectomes.csv")

missing_subjects = pd.concat([missing_subjects_functional, missing_subjects_structural]).drop_duplicates().reset_index(drop=True)

if cohort_type == "F32":
    depression_cohort_file = project_root / "data" / "UKB" / "cohorts" / "depression_cohort_F32.csv"
    exclude_subjects_by_eid(depression_cohort_file, missing_subjects, eid_col="eid", 
                            source_eid_col="subject_id", out_path=depression_cohort_file, inplace=True, return_count=False)

exclude_subjects_by_eid(combined_cohort_file, missing_subjects, eid_col="eid", 
                        source_eid_col="subject_id", out_path=combined_cohort_file, inplace=True, return_count=False)